# Enriquecimiento del Dataset

El objetivo de este notebook es el uso de las descripciones textuales de los anuncios como una fuente adicional de información para enriquecer el dataset principal.

El enriquecimiento se produce utilizando dos métodos:

* Creación de Features & Scoring: Se codificaron features binarias a partir de un diccionario de conceptos claves del ámbito inmobiliario. Además, se creó un scoring agregado, que resume en un único valor el atractivo del anuncio según la presencia de atributos positivos y negativos.


* Generación de Embeddings: se crea un embedding completo para cada descripción con el fin de mantener la mayor información semántica posible para la red neuronal.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## 1. Preprocesamiento

En el notebook anterior el enfoque de limpieza se aplicó a variables categóricas, binarias y numéricas. Como el preprocesamiento de texto requiere pasos distintos, se realiza en un apartado independiente. A continuación se explica, paso a paso, el proceso de limpieza y homogeneización.

Empezamos por revisar si hay valores nulos existentes en la columna de descripción.

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/TFM/Ficheros CSV/datos_Idealista_limpios_V_final.csv")

print(f"Verificación de valores Nulos: {df["description"].isna().sum()}")
print(f"Tamaño del dataset: {df.shape}")

df.head(3)

Verificación de valores Nulos: 38
Tamaño del dataset: (26261, 36)


,bathrooms,description,detailedType.subTypology,detailedType.typology,district,exterior,floor,has360,has3DTour,hasLift,...,priceInfo.price.priceDropInfo.priceDropValue,propertyType,rooms,showAddress,size,status,topPlus,floor_cat,has_neighborhood,has_Discount
0,1,**Exclusivo ático en el emblemático barrio de ...,penthouse,flat,Centro,1,4.0,0,1,1,...,45000.0,penthouse,2,0,106.0,good,1,Numerica,1,1
1,1,Amplio piso en el centro de Madrid Engel&Völke...,Vacío,flat,Centro,1,3.0,0,1,1,...,39000.0,flat,5,0,140.0,good,1,Numerica,1,1
2,1,Coqueta y luminosa propiedad con terraza en La...,Vacío,flat,Centro,1,4.0,0,1,1,...,45000.0,flat,2,0,106.0,good,1,Numerica,1,1


### 1.1 Limpieza y procesamiento textual

El dataset contiene 38 valores nulos en la columna descripción, lo que representa solo un 0,14% del total. Los embeddings no pueden procesar este tipo de valores, por lo que deben ser imputados. Dado el bajo número de casos, aplicar técnicas de imputación complejas no es práctico ni aporta valor significativo.
Por ello, se utiliza una sustitución simple por `No description`.

In [ ]:
df["description"] = df["description"].fillna("No description")
df["description"].isna().sum()

np.int64(0)

### 1.2 Homogenización de las Descripciones

Para garantizar una homogenización de las descripciones, se transforman las palabras a minusculas, y se eliminan caracteres especiales no relevantes para el análisis. Se mantienen algunos símbolos como `$` y `€`, ya que pueden aportar contexto adicional relevante.

La transformación a minúsculas reduce la cardinalidad del vocabulario, lo que puede disminuir la carga computacional durante el procesamiento.

In [ ]:
df["description"] = df["description"].str.replace("[^a-zA-ZñÑáéíóú .,$€'\-%:]", "", regex=True)
df["description"] = df["description"].str.lower()
df.head(3)

,bathrooms,description,detailedType.subTypology,detailedType.typology,district,exterior,floor,has360,has3DTour,hasLift,...,priceInfo.price.priceDropInfo.priceDropValue,propertyType,rooms,showAddress,size,status,topPlus,floor_cat,has_neighborhood,has_Discount
0,1,exclusivo ático en el emblemático barrio de pa...,penthouse,flat,Centro,1,4.0,0,1,1,...,45000.0,penthouse,2,0,106.0,good,1,Numerica,1,1
1,1,amplio piso en el centro de madrid engelvlkers...,Vacío,flat,Centro,1,3.0,0,1,1,...,39000.0,flat,5,0,140.0,good,1,Numerica,1,1
2,1,coqueta y luminosa propiedad con terraza en la...,Vacío,flat,Centro,1,4.0,0,1,1,...,45000.0,flat,2,0,106.0,good,1,Numerica,1,1


### 1.3 Normalización de abreviaciones

En el sector inmobiliario es frecuente usar abreviaciones para no exceder el número de caracteres y maximizar la información que se presenta en las descripciones. Aunque estas descripciones pueden ser sencillas de entender para una persona, los embeddings podrían no reconocer la relación semántica entre las abreviaciones y palabras completas.
Para que los modelos capturen correctamente estas relaciones en el espacio vectorial se normalizan las abreviaciones en dos fases:

1. Se detectan las abreviaciones más frecuentes en la descripción.
2. Se construye una lista con las abreviaciones encontradas y se añaden otras típicas del sector, garantizando una normalización más completa.


In [ ]:
import re
from collections import Counter

description = df["description"].to_list()

patterns = {
    "words": r"\b[a-zA-Z]{2,5}\.(?=\s|$)",
    "acronyms": r"\b(?:[A-Z]\.){2, }"
}


abbreviations = []
for doc in description:
  for category, pattern in patterns.items():
    abbreviations.extend(re.findall(pattern, doc))

In [ ]:
# Abreviacones más frecuentes
freq = Counter(abbreviations)
# freq.most_common(50)

In [ ]:
# Diccionario de abreviaciones comunes en el ámbito inmobiliario
abbreviations_dic = {
    "aprox.": "aproximado",
    "km.": "kilometro",
    "kms.": "kilometro",
    "m.": "metros",
    "m2": "metros cuadrados",
    "coop.": "cooperativa",
    "ref.": "referencia",
    "min.": "minutos",
    "av.": "avenida",
    "avd.": "avenida",
    "no.": "número",
    "nº": "numero",
    "iva.": "iva",
    "max.": "maximo",
    "izq": "izquierdo",
    "dpto.": "departamento",
    "hab.": "habitacion",
    "pb.": "planta baja",
    "sup.": "superficie",
    "habs.": "habitaciones",
    "ref": "referencia"
}

def normalize_description(text, abb_dict):
  for abbr, full in abb_dict.items():
        text = re.sub(rf"\b{re.escape(abbr)}\b", full, text, flags=re.IGNORECASE)
  return text

description_norm = [normalize_description(value, abbreviations_dic) for value in description]

### 1.4 Análisis de longitud de descripciones

Antes de construir el diccionario, evaluamos la longitud de cada descripción para entender su extensión y definir límites de truncado o padding para los modelos de NLP. Calculamos la longitud media y el percentil 95, que cubre la mayoría de los casos sin desperdiciar recursos.

In [ ]:
# Longitud media y extensión típica de los enunciados

import numpy as np

lengths = [len(seq) for seq in df["description"]]
print("Longitud media:", np.mean(lengths))
print("Percentil 95:", np.percentile(lengths, 95))

Longitud media: 1603.0654582841476
Percentil 95: 3204.0


## 2. Generación de Vocabulario

En esta etapa se identificaron los términos más relevantes dentro de las descripciones. La idea es construir un vocabulario controlado que pueda servir como base para enriquecer el dataset, y posteriormente crear las features.

El procedimiento combina dos enfoques:

1. Diccionario Inicial: se incluye un conjunto de términos definidos manualmente que funcionan como semilla. Este paso garantiza que no se pierdan conceptos claves del sector que podrían no aparecer con tanta frecuencia en los datos brutos.

2. Creación automática con spaCy: se procesan las descripciones usando un modelo de lenguaje entrenado en español *es_core_news_sm*.
Se aprovechan las características del modelo para:
- Filtrar sustantivos y adjetivos, ya que son los atributos que mejor describen los inmuebles.
- Se normalizan las palabras a su lema, disminuyendo la redundancia.
- Se eliminan las stopwords que no aportan valor semántica y diluyen los pesos de atributos relevantes

In [ ]:
DICT_ADJ = {
    # Positives generales
    "terraza": {"terraza", "balcón", "solárium", "patio", "roof garden", "roof", "rooftop", "tejado"},
    "atico": {"ático", "penthouse", "última planta", "loft", "ático dúplex"},
    "garaje": {"garaje", "aparcamiento", "parking", "plaza privada", "privado"},
    "ascensor": {"ascensor", "elevador"},
    "espaciosa": {"bien distribuida", "espaciosa", "amplia", "grande", "grandísima"},
    "luminoso": {"luminoso", "soleado", "claro", "brillante", "solado", "luminosa"},
    "reformado": {"reformado", "renovado", "actualizado", "rehabilitado"},
    "obra_nueva": {"obra nueva", "a estrenar", "recién construido"},
    "céntrico": {"céntrico", "bien ubicado", "zona prime", "en el centro", "center", "prime"},
    "interior": {"interior", "da a patio interior", "sin vistas", "interno"},

    # Negativos / desventajas
    "antiguo": {"antiguo", "viejo", "clásico en mal estado", "desgastado", "deteriorado", "caduca"},
    "a_reformar": {"a reformar", "necesita reforma", "a renovar", "obsoleto", "rehabilitar", "degradado"},
    "sin_ascensor": {"sin ascensor", "sin elevador"},
    "okupa": {"okupa", "ocupado", "ocupas"},
    "oscuro": {"oscuro", "poca luz", "tenebroso", "sombrío"},
    "pequeño": {"pequeño", "estrecho", "reducido", "minúsculo"},
    "mal_ubicado": {"alejado", "periferia", "lejano", "desconocido"},

    # Proximidad a transporte / servicios
    "cercano_metro": {"metro", "estación", "subway", "tren", "parada"},
    "cercano_supermercado": {"supermercado", "tienda", "market"},
    "cercano_colegio": {"colegio", "escuela", "instituto", "guardería", "kinder", "universidad", "facultad"},
    "cercano_hospital": {"hospital", "clinica", "clínica", "centro de atención primaria"},
    "cercano_farmacia": {"farmacia", "pharmacy", "botica", "droguería", "dispensario", "centro de salud"},
    "cercano_mall": {"centro comercial", "shopping mall", "mall"},
    "cercano_correos": {"oficina de correos", "correos"},
    "cercano_policia": {"comisaría", "policía", "bomberos"},
    "cercano_cultural": {"cine", "teatro", "centro cultural"},

    # Amenities
    "gimnasio": {"gimnasio", "gym", "fitness"},
    "piscina": {"piscina", "pool"},
    "padel": {"padel", "paddle"},
    "zonas_comunes": {"zona común", "club social", "salón de eventos"},
    "parque": {"parque infantil", "jardin", "zona verde"}
}


In [ ]:
!pip install spacy
!python -m spacy download es_core_news_sm

import spacy
import es_core_news_sm

nlp = spacy.load("es_core_news_sm")
nlp = es_core_news_sm.load()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 48.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# función para separar en sustantivo y adjetivo, además de lematizar las palabras.
def build_vocab (description_norm):
  vocab = set()
  tokens = []
  for sent in description_norm:
    doc = nlp (sent)
    for token in doc:
      if token.is_alpha and token.pos_ in {"NOUN", "ADJ"} and not token.is_stop:
        lemma = token.lemma_.lower().strip()
        vocab.add(lemma)
        tokens.append(lemma)
  return vocab, tokens

# ejecutamos función
vocab_spacy, tokens_spacy = build_vocab(description_norm)

KeyboardInterrupt: 

El vocabulario extraído automáticamente con la función anterior `vocab_spacy` se combinan con el diccionario manual. Esto garantiza que el vocabulario final incluya palabras frecuentes en los datos como conceptos clave del sector, eliminando duplicados y facilitando su revisión.

In [ ]:
dict_terms = {t for syns in DICT_ADJ.values() for t in syns}
ordered_vocab = sorted(vocab_spacy.union(dict_terms))

In [ ]:
drive.mount('/content/drive/')
file_path = '/content/drive/MyDrive/TFM/Ficheros CSV/_20250910_vocab.txt'

with open(file_path, "w", encoding="utf-8") as f:
    for word in ordered_vocab:
        f.write(word + "\n")

## 3. Enriquecimiento del Diccionario con embeddings

Hasta ahora el diccionario solo ha sido capaz de capturar coincidencias exactas. Para enriquecerlo con relaciones semánticas se utilizan los embeddings generados con SentenceTransformer (`paraphrase-multilingual-mpnet-base-v2`). Este modelo:

- Representa cada oración o palabra en un espacio vectorial de 768 dimensiones.

- Permite medir similitud semántica entre términos mediante cosine similarity, útil para identificar palabras relacionadas o sinónimos que no estaban explícitamente en el diccionario.

Nota: no se evalúa la precisión del modelo, ya que se utiliza un modelo preentrenado cuya eficacia en tareas de similitud semántica ya ha sido demostrada. El objetivo es generar embeddings útiles para enriquecer el diccionario. El proceso de enriquecimiento funciona así:

1. Se generan embeddings para todo el vocabulario (vocab) y opcionalmente se consideran las frecuencias de los tokens en el corpus.

2. Para cada palabra o término del diccionario manual, se buscan las k palabras más similares según la similitud de coseno.

3. Solo se agregan aquellas palabras que cumplen un umbral mínimo de similitud (sim_threshold) y, si se usa frecuencia, aparecen al menos un número mínimo de veces en el corpus.

El resultado es un diccionario enriquecido (enriched_dict) que mantiene los términos originales y añade palabras semánticamente relacionadas, capturando relaciones que antes se perdían con coincidencias literales.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer

model_1 = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

vocab = set(ordered_vocab)

def enrich_dict_phrases(dict_adj, model, vocab, corpus_tokens=None, k=3, sim_threshold=0.65):
    vocab_list = list(vocab)
    vocab_embeddings = model.encode(vocab_list, show_progress_bar=True)

    freq = {}
    if corpus_tokens:
        for token in corpus_tokens:
            freq[token] = freq.get(token, 0) + 1

    enriched = {}

    for key, terms in dict_adj.items():
        enriched[key] = set(terms)
        for term in terms:
            vector = model.encode([term])
            sims = cosine_similarity(vector, vocab_embeddings)[0]

            top_indices = np.argsort(sims)[::-1][1:k+1]

            for i in top_indices:
                candidate = vocab_list[i]
                if sims[i] < sim_threshold:
                    continue
                if corpus_tokens and freq.get(candidate, 0) < 3:
                    continue
                enriched[key].add(candidate)

    return enriched

# Se enriquece el diccionario con términos semánticos parecidos
enriched_dict = enrich_dict_phrases(DICT_ADJ, model_1, vocab, corpus_tokens=tokens_spacy, k=3, sim_threshold=0.65)

enriched_dict

### 3.1 Revisión manual del diccionario

Una vez obtenido el diccionario enriquecido, se realiza una última limpieza manual para eliminar términos mal asignados, ambiguos o fuera de contexto.
Esto garantiza que el vocabulario final sea coherente y relevante para la construcción de features y el análisis posterior.

In [ ]:
clean_enrich_dict_1 ={
    'terraza': {'balcon','balconada','balcón','balcónterraza','patio','patiojardín','patioterraza','roof','roof garden','rooftop','solarium','solárium','tejado','terraza','terrazajardín','ático'},
    'atico': {'loft', 'lofts', 'penthouse', 'penúltimo', 'ultima', 'ultimo', 'ático', 'ático dúplex', 'áticodúplex', 'última planta'},
    'garaje': {'aparcamiento','estacionamiento','garaje','parking','plaza','plaza privada','privada','privado','private'},
    'ascensor': {'ascensor', 'elevador', 'elevadoro', 'elevator'},
    'espaciosa': {'alargada','amplia','bien distribuida','distribuida','distribuido','distribuído','enorme','espaciosa','espacioso','grande','grandísima','grandísimo','spacio','stor'},
    'luminoso': {'brillante','brillo','claro','luminosa','luminoso','sol','solado','soleada','soleado','tabit'},
    'reformado': {'actualizacion','actualización','actualizada','actualizado','reformada','reformado','rehabilitado','renovacion','renovación','renovada','renovado'},
    'obra_nueva': {'a estrenar','aparición','construida','construído','estrenado','novedad','novedoso','obra nueva','recién construido'},
    'céntrico': {'bien ubicado','center','centr','centro','cifra','céntrico','en el centro','localizado','prime','primo','ubicado','zona','zona prime','zone'},
    'interior': {'ausencia','da a patio interior','interior','interno','sin vistas','terrazajardín'},
    'antiguo': {'agotado','antiguedad','antiguo','caduca','clásico','clásico en mal estado','desgastado','deteriorado','deterioro','lama','viejo'},
    'a_reformar': {'a reformar','a renovar','degradado','deterioro','innecesario','necesita reforma','obsoleto','reforma','reformista','rehabilitado','rehabilitar','renovacion','renovada'},
    'sin_ascensor': {'elevador', 'elevadoro', 'sin ascensor', 'sin elevador'},
    'okupa': {'ocupado', 'ocupas', 'okupa', 'okupado'},
    'oscuro': {'foscurit','light','luz','oscuro','poca luz','sombrío','tenebroso'},
    'pequeño': {'estratgico','estrecho','estremera','minúsculo','pequeño','rebajado','reducción','reducido'},
    'mal_ubicado': {'alamedo','alejado','alisado','desconocido','lejana','lejano','leñera','líneo','periferia','periférico','perimetral','perímetro'},
    'cercano_metro': {'estación','ferroviario','metro','metroútil','parada','parado','subway','tren'},
    'cercano_supermercado': {'market','mercado','supermercado','tienda'},
    'cercano_colegio': {'barn','colegio','conserjeria','escolar','escuela','escuelainstituto','facultad','facultativo','guarderia','guardería','infantil','institución','instituto','kinder','niño','semestre','universidad','universitario','university'},
    'cercano_hospital': {'centro de atención primaria','clinica','clinico','clínica','hospital','hospitalario'},
    'cercano_farmacia': {'botica','centro de salud','dispensario','droguería','farmacia','farmacio','pharmacy','sanitario'},
    'cercano_mall': {'centro comercial', 'mall', 'shopping mall'},
    'cercano_correos': {'correo','correos','oficina de correos','postal'},
    'cercano_policia': {'bombero','bomberos','comisaría','patrulla','policía'},
    'cercano_cultural': {'centro cultural','cine','filmoteca','película','teatral','teatro'},
    'gimnasio': {'ejercicio', 'fitness', 'gimnasio', 'gym'},
    'piscina': {'piscina', 'pool'},
    'padel': {'paddel', 'paddle', 'padel'},
    'zonas_comunes': {'club','club social','clubhome','salón','salón de eventos','social','zona común','zone'},
    'parque': {'green','jardin','jardinería','park','parque infantil','parques','zona verde'}}

## 4. Creación de Features & Scoring

La construcción de features es clave para asegurar que el scoring sea interpretable, correcto y útil. Se implementaron dos estrategias diferentes y se compararon según su precisión, métricas de clasificación y eficiencia computacional para seleccionar la más adecuada:


1. Estrategia Embedding contextual + `OneVsRestClassifier`:
- Se generan embeddings utilizando un modelo preentrenado
- Cada feature del diccionario se trata como una etiqueta binaria y se clasifica utilizando `OneVsRestClassifier` con regresión logística.

2. Estrategia Embeddings + Classificación con BERT:
- Se generan embeddings
- Se clasifican utilizando BERT
- Cada feature sigue un esquema de etiqueta binaria, pero el modelo captura relaciones semánticas más complejas.

### 4.1. Estrategia Embedding contextual + `OneVsRestClassifier`

En esta sección se construyen features binarias a partir del diccionario enriquecido y se entrenan modelos para predecir la presencia de cada característica en las descripciones. El objetivo es generar un dataset de features interpretables, que luego servirá para calcular el scoring de cada anuncio.

La estrategia combina:

Embeddings de las descripciones usando un modelo preentrenado `SentenceTransformer` que captura la información semántica del texto.

Clasificación de cada feature como etiqueta binaria usando OneVsRestClassifier con regresión logística, optimizando hiperparámetros mediante GridSearchCV.

Evaluación del modelo con métricas generales `F1-score macro/micro`, `accuracy`, `Hamming Loss` y métricas individuales por feature para asegurar la calidad de las predicciones.

El resultado final es un dataset de features binarios, donde cada columna representa un atributo del diccionario y cada fila un anuncio.


Estrategia 1: Embedding contextual + OneVsRestClassifier – Pipeline

Generación de etiquetas binarias

Cada descripción se convierte en una fila de etiquetas 1/0 según la presencia de los términos del diccionario enriquecido.

Creación de embeddings de las descripciones

Se usan SentenceTransformer (paraphrase-multilingual-mpnet-base-v2) para generar embeddings que capturan la información semántica de cada anuncio.

Entrenamiento del clasificador OneVsRest

Se utiliza OneVsRestClassifier con regresión logística, optimizando hiperparámetros mediante GridSearchCV.

Cada feature se trata como una etiqueta independiente.

Evaluación del modelo

Se calculan métricas globales (F1 macro/micro, accuracy, Hamming Loss) y métricas por feature para analizar la calidad de la predicción.

Predicción de features para todo el dataset

Se genera un dataset de features binarias, con cada columna representando un atributo y cada fila un anuncio, listo para scoring y análisis posterior.


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, make_scorer
from sentence_transformers import SentenceTransformer

1. Se generación etiquetas binarias marcando `1` si alguna palabra coincide con el diccionario y `0`en caso contrario.

In [ ]:
def label_descriptions(descriptions, enriched_dict):
    labels = []
    for desc in descriptions:
        row = []
        desc_lower = desc.lower()
        for key, synonyms in enriched_dict.items():
            row.append(int(any(s in desc_lower for s in synonyms)))
        labels.append(row)
    return np.array(labels)


2. Se genera el embedding utilizando el modelo y el diccionario para entrenarlo. Devolviendo las etiquetas binarias listas para el clasificador

In [ ]:
def generate_features(description_list, enriched_dict, model_name="paraphrase-multilingual-mpnet-base-v2"):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(description_list, show_progress_bar=True)

    y = label_descriptions(description_list, enriched_dict)

    return embeddings, y, model

embeddings_1, y, model_1 = generate_features(description_norm, clean_enrich_dict_1)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/821 [00:00<?, ?it/s]

3. Se entrena el clasificador utilizando GridSearchCV, probando así varios hiperparametros y eligiendo el más óptimo.

In [ ]:
def train_with_gridsearch(X, y, param_grid, test_size=0.2):
    """
    Trains a OneVsRestClassifier with LogisticRegression using GridSearchCV.

    Returns:
    - best_clf: The best trained model from the grid search
    - X_test, y_test: The test set for final evaluation
    """
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

    clf = OneVsRestClassifier(LogisticRegression(max_iter=500, random_state=42))

    grid_search = GridSearchCV(
        estimator=clf,
        param_grid=param_grid,
        scoring=make_scorer(f1_score, average='macro'),
        cv=3,
        verbose=1,
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)

    best_clf = grid_search.best_estimator_

    print("\n--- Resultados de la Búsqueda ---")
    print(f"Mejores hiperparámetros: {grid_search.best_params_}")
    print(f"Mejor score F1 en validación cruzada: {grid_search.best_score_:.3f}")

    return best_clf, X_test, y_test

In [ ]:
param_grid = {
      'estimator__C': [5, 10, 50],
      'estimator__class_weight': ['balanced'],
      'estimator__solver': ['liblinear']
      }


best_clf, X_test, y_test = train_with_gridsearch(embeddings_1, y, param_grid)

Fitting 3 folds for each of 3 candidates, totalling 9 fits

--- Resultados de la Búsqueda ---
Mejores hiperparámetros: {'estimator__C': 50, 'estimator__class_weight': 'balanced', 'estimator__solver': 'liblinear'}
Mejor score F1 en validación cruzada: 0.539


In [ ]:
# borrar esto luego
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier

def train_best_model(X, y, test_size=0.2):
    """
    Entrena un OneVsRestClassifier con LogisticRegression
    usando los mejores hiperparámetros encontrados previamente.

    Returns:
    - best_clf: Modelo entrenado con los hiperparámetros óptimos
    - X_test, y_test: Conjunto de prueba para evaluación final
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    best_clf = OneVsRestClassifier(
        LogisticRegression(
            C=50,
            class_weight="balanced",
            solver="liblinear",
            max_iter=500,
            random_state=42
        )
    )

    best_clf.fit(X_train, y_train)

    print("\n--- Entrenamiento Final ---")
    print("Modelo entrenado con los mejores hiperparámetros.")

    return best_clf, X_test, y_test

best_clf, X_test, y_test = train_best_model(embeddings_1, y)


--- Entrenamiento Final ---
Modelo entrenado con los mejores hiperparámetros.


4. Se analizan las métricas del modelo más óptimo para luego ser comparados con la otra estrategia.

In [ ]:
from sklearn.metrics import make_scorer, f1_score, accuracy_score, hamming_loss

def evaluate_metrics(best_clf, X_test, y_test, label_names):
    """
    Evaluates the model and prints a comprehensive set of metrics.
    """
    # 1. Realizar predicciones
    y_test_pred = best_clf.predict(X_test)

    # 2. Calcular métricas generales
    f1_macro = f1_score(y_test, y_test_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_test, y_test_pred, average='micro', zero_division=0)
    acc_overall = accuracy_score(y_test, y_test_pred)
    hl = hamming_loss(y_test, y_test_pred)

    print("\n--- Desempeño General del Modelo en el Conjunto de Prueba ---")
    print(f"F1-Score (Macro): {f1_macro:.3f}")
    print(f"F1-Score (Micro): {f1_micro:.3f}")
    print(f"Accuracy: {acc_overall:.3f}")
    print(f"Hamming Loss: {hl:.3f}")

    # 3. Calcular métricas por etiqueta
    print("\n--- Métricas por Etiqueta ---")
    for i, key in enumerate(label_names):
        f1_label = f1_score(y_test[:, i], y_test_pred[:, i], average='binary', zero_division=0)
        acc_label = accuracy_score(y_test[:, i], y_test_pred[:, i])

        print(f"Etiqueta: {key}")
        print(f"  - F1-Score: {f1_label:.3f}")
        print(f"  - Accuracy: {acc_label:.3f}")

In [ ]:
label_names = list(clean_enrich_dict_1.keys())
evaluate_metrics(best_clf, X_test, y_test, label_names)


--- Desempeño General del Modelo en el Conjunto de Prueba ---
F1-Score (Macro): 0.555
F1-Score (Micro): 0.682
Accuracy: 0.025
Hamming Loss: 0.203

--- Métricas por Etiqueta ---
Etiqueta: terraza
  - F1-Score: 0.785
  - Accuracy: 0.759
Etiqueta: atico
  - F1-Score: 0.502
  - Accuracy: 0.778
Etiqueta: garaje
  - F1-Score: 0.802
  - Accuracy: 0.773
Etiqueta: ascensor
  - F1-Score: 0.718
  - Accuracy: 0.827
Etiqueta: espaciosa
  - F1-Score: 0.845
  - Accuracy: 0.771
Etiqueta: luminoso
  - F1-Score: 0.760
  - Accuracy: 0.699
Etiqueta: reformado
  - F1-Score: 0.717
  - Accuracy: 0.828
Etiqueta: obra_nueva
  - F1-Score: 0.539
  - Accuracy: 0.775
Etiqueta: céntrico
  - F1-Score: 0.901
  - Accuracy: 0.831
Etiqueta: interior
  - F1-Score: 0.496
  - Accuracy: 0.716
Etiqueta: antiguo
  - F1-Score: 0.535
  - Accuracy: 0.792
Etiqueta: a_reformar
  - F1-Score: 0.796
  - Accuracy: 0.848
Etiqueta: sin_ascensor
  - F1-Score: 0.352
  - Accuracy: 0.900
Etiqueta: okupa
  - F1-Score: 0.645
  - Accuracy: 0.

5. Se genera un dataset final de features binarias con cada columna como atributo y cada fila como enunciado

In [ ]:
features_pred_1 = best_clf.predict(embeddings_1)

features_df_1 = pd.DataFrame(features_pred_1, columns=label_names)
features_df_1.head()

,terraza,atico,garaje,ascensor,espaciosa,luminoso,reformado,obra_nueva,céntrico,interior,...,cercano_farmacia,cercano_mall,cercano_correos,cercano_policia,cercano_cultural,gimnasio,piscina,padel,zonas_comunes,parque
0,1,1,1,1,1,1,0,0,1,1,...,0,1,1,0,1,0,0,0,1,0
1,0,0,1,1,1,1,0,0,1,1,...,1,1,1,0,1,0,0,0,1,1
2,1,1,1,0,1,1,0,1,1,1,...,0,0,0,0,1,0,0,1,1,0
3,1,1,1,1,1,1,0,0,1,1,...,1,1,1,0,1,1,1,1,1,1
4,1,1,1,0,0,1,1,1,0,1,...,0,1,1,0,1,1,1,1,1,1


In [ ]:
drive.mount('/content/drive/')
# Copia para no tener que ejecutar todo el modelo nuevamente
file_path = '/content/drive/MyDrive/TFM/Ficheros CSV/_20250910_features_df_1.csv'

features_df_1.to_csv(file_path, index=False)

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).



### 4.2 Multi-label Transformer Classifier

Para superar las limitaciones de los enfoques anteriores, en la segunda estrategia se utilizan los `Embeddings`y `Transformers` para generar un clasificador multietiquetas sobre las descripciones utilizando DistilBert que es capaz de capturar relaciones semánticas más complejas y generalizar el lenguaje. Se explica a continuación el pipeline.

In [ ]:
!pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 99.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.0
    Uninstalling transformers-4.56.0:
      Successfully uninstalled transformers-4.56.0



1. **Generación de etiquetas binarias**

   * Cada descripción se convierte en una fila de etiquetas `1`/`0` según la presencia de las palabras del diccionario enriquecido.

In [ ]:
labels = label_descriptions(description_norm, clean_enrich_dict_1)

labels_list = labels.astype(float).tolist()


In [ ]:
import transformers
from datasets  import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch



2. **Creación del dataset Hugging Face**

   * Se combinan los textos y las etiquetas binarias en un dataset compatible con Transformers.

In [ ]:
# Dataset de Hugging Face
dataset = Dataset.from_dict({"text": description_norm, "labels": labels_list})

# Modelo elegido
model_name = "distilbert-base-multilingual-cased"

# Inicialización del tokenizador
tokenizer = AutoTokenizer.from_pretrained(model_name)
num_labels = labels.shape[1]

# Inicialización del modelo
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

train_test_split = dataset.train_test_split(test_size=0.2)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


3. **Tokenización y preparación de entradas**

   * Los textos se tokenizan, truncan y rellenan al tamaño máximo del modelo.
   * Las etiquetas se transforman en tensores flotantes para el entrenamiento.


In [ ]:
def tokenize_and_format_labels(batch):
    """Tokeniza el texto y formatea las etiquetas para el entrenamiento."""
    tokenized_batch = tokenizer(batch["text"], truncation=True, padding=True, max_length=128)
    tokenized_batch["labels"] = [torch.tensor(l, dtype=torch.float32) for l in batch["labels"]]
    return tokenized_batch


In [ ]:
# Mapeop del tokenizador
train_dataset = train_test_split["train"].map(tokenize_and_format_labels, batched=True)
test_dataset = train_test_split["test"].map(tokenize_and_format_labels, batched=True)

Map:   0%|          | 0/21008 [00:00<?, ? examples/s]

Map:   0%|          | 0/5253 [00:00<?, ? examples/s]

4. **Inicialización del modelo Transformer**

   * Se utiliza `distilbert-base-multilingual-cased` como modelo preentrenado para clasificación multi-etiqueta.

5. **Definición del entrenamiento con Trainer**

   * Se configuran parámetros como epochs, batch size, learning rate y weight decay.
   * Se calculan métricas de desempeño globales (F1 macro/micro, accuracy, Hamming Loss) y por feature individual.

In [ ]:
# Configuración de los parametros
from transformers.trainer_utils import IntervalStrategy

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    num_train_epochs= 5, # se probo con 3 y el modelo podía mejorar -> se sube a 5
    learning_rate= 2e-5, # se agregaron métricas
    weight_decay= 0.01, # se agregaron métricas
    warmup_ratio= 0.06, # se agregaron métricas
    report_to="none",
    )

trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=train_dataset,
  eval_dataset=test_dataset,
  tokenizer=tokenizer,
  compute_metrics = compute_metrics
  )

trainer.train()
trainer.save_model("./modelo-features-embeddings")
!cp -r "./modelo-features-embeddings "/content/drive/MyDrive/TFM/Ficheros CSV/"


Step,Training Loss
500,0.485500
1000,0.368500
1500,0.328100
2000,0.310100
2500,0.292400
3000,0.277800
3500,0.270200
4000,0.264000
4500,0.249300
5000,0.248400


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


6. **Entrenamiento del modelo**

   * El modelo aprende a asociar la información semántica del texto con la presencia de cada feature.




In [ ]:
def compute_metrics(eval_pred):
    """
    Calcula las métricas de evaluación: F1 (macro y micro), Accuracy y Hamming Loss.
    """
    logits, labels = eval_pred

    # Se aplica la función sigmoide para obtener probabilidades y se binariza
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    y_pred = (probs >= 0.5).astype(int)

    # Asegurarse de que labels y y_pred tengan el mismo dtype
    labels = labels.astype(int)

    # Cálculo de métricas
    f1_macro = f1_score(labels, y_pred, average="macro", zero_division=0)
    f1_micro = f1_score(labels, y_pred, average="micro", zero_division=0)
    hamming_loss_val = hamming_loss(labels, y_pred)
    accuracy_val = accuracy_score(labels, y_pred)

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "accuracy": accuracy_val,
        "hamming_loss": hamming_loss_val,
    }

In [ ]:
print("\n--- Evaluación final ---")

# Realiza la evaluación en el conjunto de prueba
eval_results = trainer.evaluate()

# Imprime los resultados
print("Resultados de la Evaluación:")
for metric_name, value in eval_results.items():
    print(f"- {metric_name}: {value:.4f}")


--- Evaluación final ---


Resultados de la Evaluación:
- eval_loss: 0.2496
- eval_f1_macro: 0.5746
- eval_f1_micro: 0.8165
- eval_accuracy: 0.0952
- eval_hamming_loss: 0.0999
- eval_runtime: 20.3359
- eval_samples_per_second: 258.3120
- eval_steps_per_second: 32.3070
- epoch: 5.0000


7. **Evaluación y predicción**

   * Se generan predicciones sobre el conjunto de prueba.
   * Se calculan métricas generales y por etiqueta para validar el desempeño del modelo.



In [ ]:
# version 2

full_tokenized_dataset = dataset.map(tokenize_and_format_labels, batched=True)

# Realizar predicciones en todo el dataset
full_predictions_output = trainer.predict(full_tokenized_dataset)

# Obtener los logits
full_logits = full_predictions_output.predictions

import torch

# Convertir logits en probabilidades aplicando la función sigmoide
full_probabilities = torch.sigmoid(torch.tensor(full_logits)).numpy()

# Binarizar las probabilidades
full_predicted_labels = (full_probabilities >= 0.5).astype(int)

Map:   0%|          | 0/26261 [00:00<?, ? examples/s]

8. **Consolidación de resultados**

   * Se crean DataFrames con etiquetas reales, predichas y textos originales para análisis y revisión de resultados.

In [ ]:
import pandas as pd

label_names = list(clean_enrich_dict_1.keys())


# Create a DataFrame from your original descriptions
df_original = pd.DataFrame({"text": description_norm})

# Create a DataFrame from the full set of predicted labels
full_predicted_labels_df = pd.DataFrame(full_predicted_labels, columns=[f"{name}" for name in label_names])

# Concatenate the two DataFrames after resetting their indices
df_with_predictions = pd.concat([df_original.reset_index(drop=True), full_predicted_labels_df.reset_index(drop=True)], axis=1)


df_with_predictions.head()

,text,terraza,atico,garaje,ascensor,espaciosa,luminoso,reformado,obra_nueva,céntrico,...,cercano_farmacia,cercano_mall,cercano_correos,cercano_policia,cercano_cultural,gimnasio,piscina,padel,zonas_comunes,parque
0,exclusivo ático en el emblemático barrio de pa...,1,1,1,1,1,1,0,1,1,...,0,1,0,0,0,0,0,0,1,0
1,amplio piso en el centro de madrid engelvlkers...,1,0,1,1,1,1,0,0,1,...,0,0,0,0,0,0,0,0,1,0
2,coqueta y luminosa propiedad con terraza en la...,1,0,1,0,1,1,0,1,1,...,0,0,0,0,0,0,0,0,1,0
3,"gilmar consulting inmobiliario, ofrece en excl...",1,0,1,0,1,1,0,0,1,...,0,0,0,0,0,0,1,0,1,1
4,"exclusivo bajo con jardín de esquina, con pisc...",1,0,1,0,1,1,0,0,1,...,0,0,0,0,0,1,1,0,1,1


In [ ]:

drive.mount('/content/drive/')
file_path = '/content/drive/MyDrive/TFM/Ficheros CSV/_20250910_features_df_2.csv'

df_with_predictions.to_csv(file_path, index=False)

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    y_pred = (probs >= 0.5).astype(int)

    # Métricas globales
    f1_macro = f1_score(labels, y_pred, average="macro", zero_division=0)
    f1_micro = f1_score(labels, y_pred, average="micro", zero_division=0)
    hamming_loss_val = hamming_loss(labels, y_pred)
    accuracy_val = accuracy_score(labels, y_pred)

    metrics = {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "accuracy": accuracy_val,
        "hamming_loss": hamming_loss_val,
    }

    # Métricas por etiqueta
    for i, name in enumerate(label_names):
        f1_label = f1_score(labels[:, i], y_pred[:, i], average="binary", zero_division=0)
        acc_label = accuracy_score(labels[:, i], y_pred[:, i])
        metrics[f"f1_{name}"] = f1_label
        metrics[f"acc_{name}"] = acc_label

    return metrics

### 4.3 Comparación de los Modelos Clasificadores.
La tabla resume las métricas obtenidas para cada estrategia. La **estrategia 2** se muestra superior en todas las métricas comparadas.


| Métrica      | Estrategia 1 | Estrategia 2 |
| ------------ | ------------ | ------------ |
| F1 Macro     | 0.555        | 0.575        |
| F1 Micro     | 0.682        | 0.817        |
| Accuracy     | 0.025        | 0.095        |
| Hamming Loss | 0.203        | 0.100        |


#### 4.3.1 Interpretación de las Métricas obtenidas en la estrategia 2

**F1 Score (Macro vs. Micro)**

La métrica F1 explica la precisión y el recall de un modelo. La precisión se refiere a qué porcentaje de etiquetas predichas como positivas son correctas. Recall explica qué porcentaje de etiquetas reales positivas de detectaron.

*  El F1 Macro: calcula el F1 y luego hace un promedio. Es normal que este valor sea bajo, ya que depende de la cantidad de apariciones de dicha etiqueta en el dataset, haciendo que la media varie y tienda ir más hacia abajo.
* El F1 Micro: calcula F1 considerando todas las etiquetas y muestras juntas, ponderando más las etiquetas frecuentes. Por eso, un F1 Micro alto refleja un buen rendimiento global del modelo.

**Accuracy**

Mide el porcentaje de filas en las que todas las etiquetas fueron predichas correctamente. Acertar todas las etiquetas multiples simultáneamente en este tipo de modelos es complejo, por eso este valor se muestra muy bajo para ambos modelos.  
Si se quisiera demostrar la precisión de las etiquetas, habría que hacer un estudio indivual para cada etiqueta. Un valor bajo para modelos multi-etiquetas es de esperar.

**Hamming Loss**

Esta métrica cuantifica la fracción de etiquetas predichas incorrectamente en un problema de clasificación multietiqueta. Un valor más bajo indica que el modelo comete menos errores por etiqueta, aunque no necesariamente acierte todas las etiquetas de cada muestra.




## 5. Scoring

Se crea un diccionario con los pesos, elegidos según importancia relativa.

In [ ]:
feat_weights = {
    "terraza": 0,
    "atico": 1,
    "garaje": 2,
    "ascensor": 3,
    "espaciosa": 2,
    "luminoso": 3,
    "reformado": 2,
    "obra_nueva": 3,
    "céntrico": 1,
    "interior": -1,
    "parque": 1,
    "cercano_metro": 3,
    "cercano_supermercado": 3,
    "cercano_colegio": 2,
    "cercano_hospital": 3,
    "antiguo": -1,
    "a_reformar": 2,
    "sin_ascensor": -1,
    "okupa": -3,
    "oscuro": -3,
    "pequeño": 2,
    "mal_ubicado": -2,
    "cercano_farmacia": 1,
    "cercano_mall": 1,
    "cercano_correos": 1,
    "cercano_policia": 1,
    "cercano_cultural": 1,
    "gimnasio": 1,
    "piscina": 1,
    "padel": 1,
    "zonas_comunes": 1,
}

weight_series = pd.Series(feat_weights)


In [ ]:
df_with_predictions.columns

Index(['text', 'terraza', 'atico', 'garaje', 'ascensor', 'espaciosa',
       'luminoso', 'reformado', 'obra_nueva', 'céntrico', 'interior',
       'antiguo', 'a_reformar', 'sin_ascensor', 'okupa', 'oscuro', 'pequeño',
       'mal_ubicado', 'cercano_metro', 'cercano_supermercado',
       'cercano_colegio', 'cercano_hospital', 'cercano_farmacia',
       'cercano_mall', 'cercano_correos', 'cercano_policia',
       'cercano_cultural', 'gimnasio', 'piscina', 'padel', 'zonas_comunes',
       'parque'],
      dtype='object')

Utilizando el dataframe creado en la sección anterior, se crea una columan `` scoring_final` que sumara todas las features con su peso asignado.

In [ ]:
df_features_scoring_ = df_with_predictions.copy()
# df_features_scoring_.drop('text', axis='columns', inplace=True)

def compute_score(row):
    score = 0
    for col, weight in weight_series.items():
        if row[col] == 1:
            score += weight
    return score

df_features_scoring_["score_final"] = df_features_scoring_.apply(compute_score, axis=1)

df_features_scoring_.head()

,text,terraza,atico,garaje,ascensor,espaciosa,luminoso,reformado,obra_nueva,céntrico,...,cercano_mall,cercano_correos,cercano_policia,cercano_cultural,gimnasio,piscina,padel,zonas_comunes,parque,score_final
0,exclusivo ático en el emblemático barrio de pa...,1,1,1,1,1,1,0,1,1,...,1,0,0,0,0,0,0,1,0,21
1,amplio piso en el centro de madrid engelvlkers...,1,0,1,1,1,1,0,0,1,...,0,0,0,0,0,0,0,1,0,21
2,coqueta y luminosa propiedad con terraza en la...,1,0,1,0,1,1,0,1,1,...,0,0,0,0,0,0,0,1,0,19
3,"gilmar consulting inmobiliario, ofrece en excl...",1,0,1,0,1,1,0,0,1,...,0,0,0,0,0,1,0,1,1,22
4,"exclusivo bajo con jardín de esquina, con pisc...",1,0,1,0,1,1,0,0,1,...,0,0,0,0,1,1,0,1,1,14


Se une el dataframe creado con el original para utilizarlo en los futuros de Clustering y Deep Learning.

In [ ]:
df_features_final_model = pd.concat([df, df_features_scoring_], axis=1)
df_features_final_model.drop("description", axis="columns", inplace=True)
df_features_final_model.head()

,bathrooms,detailedType.subTypology,detailedType.typology,district,exterior,floor,has360,has3DTour,hasLift,hasPlan,...,cercano_mall,cercano_correos,cercano_policia,cercano_cultural,gimnasio,piscina,padel,zonas_comunes,parque,score_final
0,1,penthouse,flat,Centro,1,4.0,0,1,1,1,...,1,0,0,0,0,0,0,1,0,21
1,1,Vacío,flat,Centro,1,3.0,0,1,1,1,...,0,0,0,0,0,0,0,1,0,21
2,1,Vacío,flat,Centro,1,4.0,0,1,1,1,...,0,0,0,0,0,0,0,1,0,19
3,3,Vacío,flat,Encinar de los Reyes,1,0.0,0,1,1,1,...,0,0,0,0,0,1,0,1,1,22
4,4,Vacío,flat,Zona Prado de Somosaguas - La Finca,1,0.0,0,1,1,1,...,0,0,0,0,1,1,0,1,1,14


In [ ]:

drive.mount('/content/drive/')
file_path = '/content/drive/MyDrive/TFM/Ficheros CSV/_20250910_dataset_features_model.csv'

df_features_final_model.to_csv(file_path, index=False)

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


# Embedding + PCA

Se crea un embedding aparte que no entra en contacto con el diccioanrio enriquecido, así mantenemos la totalidad de la semantica de la descripción.

In [ ]:
from sentence_transformers import SentenceTransformer

model_2 = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
# embeddings
embeddings_2 = model_2.encode(description_norm,  show_progress_bar=True)
np.save("embeddings_2.npy", embeddings_2)

In [ ]:
pca = PCA(n_components=None)
pca.fit(embeddings_2)

Batches:   0%|          | 0/546 [00:00<?, ?it/s]

In [ ]:
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--')
plt.title('Explained Variance vs. Number of Components')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.grid(True)
plt.axhline(y=0.95, color='r', linestyle='-')
plt.text(0.5, 0.95, '95% threshold', color='red', transform=plt.gca().transAxes)
plt.show()


In [ ]:
# ELEGIR NÜMERO
n_components_final =

pca = PCA(n_components=n_components_final)
principal_components = pca.fit_transform(embeddings_2)

pca_df = pd.DataFrame(data=principal_components, columns=pc_columns)

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

embeddings = np.load("embeddings_1.npy")

pca = PCA(n_components=2)
principal_components = pca.fit_transform(embeddings)

